In [87]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint, solve_ivp
import warnings
warnings.filterwarnings('ignore')

# Stefan-Boltzmann constant
sigma = 5.67e-8  # W/(m²·K⁴)

# Transient Cooling Analysis: Nested Insulated Boxes with Graphite Thermal Storage

This notebook models the transient cooling of a system consisting of:
- Inner: 5.4 m³ of graphite thermal storage
- Layer 1: Inner insulation box (filled with Argon)
- Layer 2: Outer insulation box (filled with Argon)
- Boundary: Natural convection cooling to ambient

We track four temperatures:
1. Graphite core temperature (T_graphite)
2. Inner insulation box wall temperature (T_inner_box)
3. Outer insulation box inner wall (T_outer_inner)
4. Outer insulation box outer wall (T_outer_outer)

In [88]:
# ============================================================================
# SECTION 1: Material Properties and Functions
# ============================================================================
def gr_rho():
    #storage graphite density
    #return 1743
    return 1500
def gr_emis(temp: float) -> float:
    """Graphite emissivity as function of temperature."""
    return 0.679 + 6e-5 * temp

def gr_cp(temp):
    """Graphite specific heat capacity [J/(kg·K)]."""
    cp_arr = np.array([1.27, 1.92, 1.99, 2.13])
    temp_C = np.array([300, 750, 1200, 1995])
    temp_C = np.append(temp_C, 2400)
    cp_arr = np.append(cp_arr, 2.13)
    temp_C_interp = temp - 273.15
    if temp_C_interp > 2400:
        temp_C_interp = 2400
    if temp_C_interp < 300:
        temp_C_interp = 300
    return np.interp(temp_C_interp, temp_C, cp_arr) * 1000  # Convert to J/(kg·K)

def gr_tcx(temp):
    """Graphite thermal conductivity [W/(m·K)]."""
    temp_C = np.array([300, 750, 1200, 1995])
    conductivity = np.array([31.5, 28.7, 24.8, 23.1])
    temp_C = np.append(temp_C, 2400)
    conductivity = np.append(conductivity, conductivity[-1] + (conductivity[-1] - conductivity[-2]) / 
                           (temp_C[-2] - temp_C[-3]) * (temp_C[-1] - temp_C[-2]))
    temp_C_interp = temp - 273.15
    if temp_C_interp > 2400:
        temp_C_interp = 2400
    if temp_C_interp < 300:
        temp_C_interp = 300
    return np.interp(temp_C_interp, temp_C, conductivity)

def ins_tcx(temp):
    """Insulation thermal conductivity in Argon [W/(m·K)]."""
    temps = np.array([0, 400, 800, 1000, 2000, 2500])
    tcx = np.array([0.09, 0.09, 0.176, 0.35, 0.94, 0.94 + (0.94 - 0.35) * 500 / 1000])
    temp_C = temp - 273.15
    if temp_C < 0:
        temp_C = 0
    if temp_C > 2500:
        temp_C = 2500
    return np.interp(temp_C, temps, tcx)

def ins_cp(temp):
    """Insulation specific heat capacity [J/(kg·K)]."""
    return gr_cp(temp)

def ins_emis():
    """Insulation emissivity."""
    return 0.8

def wall_emis():
    """Outer box wall emissivity."""
    return 0.3

def wall_tcx():
    """Outer box wall thermal conductivity [W/(m·K)]."""
    return 0.018

def wall_cp(temp):
    """Insulation specific heat capacity [J/(kg·K)]."""
    return gr_cp(temp)

In [89]:
# ============================================================================
# SECTION 2: CONFIGURATION CELL - UPDATE GEOMETRIC & INITIAL CONDITIONS HERE
# ============================================================================

# *** GEOMETRIC PARAMETERS ***
V_graphite = 5.4  # m³ - Volume of graphite thermal storage
L_graphite = V_graphite ** (1/3)  # m - Side length (assuming cubic)

th_inner_insulation = 0.305  # m - Inner insulation wall thickness (between graphite and inner box)
th_outer_insulation = 0.15  # m - Outer insulation wall thickness (between inner and outer box)

# Box dimensions (calculated from geometry)
L_inner_box = 4.16  # Inner container outer dimension
L_outer_box = 6.096  # Outer container outer dimension


# Surface areas for fully enclosed boxes
A_graphite = 6 * L_graphite ** 2  # m² - Graphite surface (all 6 faces)
A_inner_inner = 6 * (L_inner_box - 2 * th_inner_insulation) ** 2  # m² - Inner box inner surface
A_inner_outer = 6 * L_inner_box ** 2  # m² - Inner box outer surface
A_outer_inner = 6 * (L_inner_box - 2 * th_outer_insulation) ** 2  # m² - Outer box inner surface
A_outer_outer = 6 * L_outer_box ** 2  # m² - Outer box outer surface

# *** MATERIAL PROPERTIES ***
rho_graphite_storage = gr_rho()  # kg/m³
m_graphite = V_graphite * rho_graphite_storage  # kg - Graphite mass

# Insulation material properties (assume similar density to insulation)
rho_insulation = 200  # kg/m³
m_inner_insulation = (L_inner_box**3 - (L_inner_box - th_inner_insulation)**3) * rho_insulation  # kg
m_outer_insulation = (L_outer_box**3 - (L_outer_box - th_outer_insulation)**3) * rho_insulation  # kg

# *** INITIAL CONDITIONS ***
T_ambient = 300  # K - Surrounding environment temperature (27°C)
T_initial_graphite = 2673  # K - Initial graphite temperature (1427°C)
T_initial_inner_inner = 2573  # K - Initial inner box inner surface temperature
T_initial_inner_outer = 350  # K - Initial inner box inner surface temperature
T_initial_outer_inner = 330  # K - Initial outer box inner surface temperature
T_initial_outer_outer = 320  # K - Initial outer box outer surface temperature

# *** DISPLAY CONFIGURATION ***
print("=" * 80)
print("SYSTEM CONFIGURATION")
print("=" * 80)
print(f"\nGraphite Storage:")
print(f"  Volume: {V_graphite:.2f} m³")
print(f"  Equivalent cube side: {L_graphite:.3f} m")
print(f"  Surface area: {A_graphite:.2f} m²")
print(f"  Density: {rho_graphite_storage:.1f} kg/m³")
print(f"  Mass: {m_graphite:.0f} kg")

print(f"\nInner Insulation Box:")
print(f"  Wall thickness: {th_inner_insulation*1000:.1f} mm")
print(f"  Outer dimension: {L_inner_box:.3f} m")
print(f"  Outer surface area: {A_inner_outer:.2f} m²")
print(f"  Mass: {m_inner_insulation:.0f} kg")

print(f"\nOuter Insulation Box:")
print(f"  Wall thickness: {th_outer_insulation*1000:.1f} mm")
print(f"  Outer dimension: {L_outer_box:.3f} m")
print(f"  Inner surface area: {A_outer_inner:.2f} m²")
print(f"  Outer surface area: {A_outer_outer:.2f} m²")
print(f"  Mass: {m_outer_insulation:.0f} kg")

print(f"\nInitial Temperatures:")
print(f"  Graphite: {T_initial_graphite:.0f} K ({T_initial_graphite-273.15:.0f}°C)")
print(f"  Inner box inner surface: {T_initial_inner_inner:.0f} K ({T_initial_inner_inner-273.15:.0f}°C)")
print(f"  Outer inner surface: {T_initial_outer_inner:.0f} K ({T_initial_outer_inner-273.15:.0f}°C)")
print(f"  Outer outer surface: {T_initial_outer_outer:.0f} K ({T_initial_outer_outer-273.15:.0f}°C)")
print(f"  Ambient: {T_ambient:.0f} K ({T_ambient-273.15:.0f}°C)")
print("=" * 80)
print()

SYSTEM CONFIGURATION

Graphite Storage:
  Volume: 5.40 m³
  Equivalent cube side: 1.754 m
  Surface area: 18.47 m²
  Density: 1500.0 kg/m³
  Mass: 8100 kg

Inner Insulation Box:
  Wall thickness: 305.0 mm
  Outer dimension: 4.160 m
  Outer surface area: 103.83 m²
  Mass: 2940 kg

Outer Insulation Box:
  Wall thickness: 150.0 mm
  Outer dimension: 6.096 m
  Inner surface area: 89.40 m²
  Outer surface area: 222.97 m²
  Mass: 3263 kg

Initial Temperatures:
  Graphite: 2673 K (2400°C)
  Inner box inner surface: 2573 K (2300°C)
  Outer inner surface: 330 K (57°C)
  Outer outer surface: 320 K (47°C)
  Ambient: 300 K (27°C)



In [90]:
# ============================================================================
# SECTION 5: Radiation Heat Transfer Functions
# ============================================================================

def heat_radiation(T_hot, T_cold, A, emis_hot, emis_cold=None, view_factor=1.0):
    """
    Calculate radiative heat transfer using Stefan-Boltzmann equation.
    
    For parallel enclosures (fully enclosed boxes):
    Q_rad = A * sigma * (T_hot^4 - T_cold^4) / (1/emis_hot + 1/emis_cold - 1)
    
    For single surface to surroundings:
    Q_rad = emis * A * sigma * (T_hot^4 - T_cold^4)
    
    Parameters:
    -----------
    T_hot : float
        Hot surface temperature (K)
    T_cold : float
        Cold surface temperature (K)
    A : float
        Surface area (m²)
    emis_hot : float
        Emissivity of hot surface (0-1)
    emis_cold : float
        Emissivity of cold surface (0-1), if None uses emis_hot
    view_factor : float
        View factor (0-1), default 1.0 for fully enclosed
    
    Returns:
    --------
    float
        Radiative heat transfer rate (W)
    """
    if emis_cold is None:
        # Single surface radiation (to surroundings)
        Q = emis_hot * A * sigma * (T_hot**4 - T_cold**4) * view_factor
    else:
        # Parallel enclosure radiation with two different emissivities
        # Use radiosity method for enclosure
        emissivity_factor = 1.0 / (1.0/emis_hot + 1.0/emis_cold - 1.0)
        Q = A * sigma * emissivity_factor * (T_hot**4 - T_cold**4) * view_factor
    
    return Q

def linearized_radiation_coefficient(T_surface, T_reference, emis):
    """
    Linearize radiation around reference temperature for small ΔT.
    
    Useful for iterative solving: h_rad = 4 * emis * sigma * T^3
    """
    T_mean = (T_surface + T_reference) / 2
    h_rad = 4 * emis * sigma * T_mean**3
    return h_rad

def natural_convection_coefficient(T_surface, T_ambient, characteristic_length, orientation='vertical'):
    """
    Calculate natural convection heat transfer coefficient.
    
    Uses Nusselt number correlations for vertical surfaces in air.
    h = k/L * Nu where Nu depends on Rayleigh number
    
    Parameters:
    -----------
    T_surface : float
        Surface temperature (K)
    T_ambient : float
        Ambient temperature (K)
    characteristic_length : float
        Characteristic length scale (m)
    orientation : str
        'vertical' or 'horizontal'
    
    Returns:
    --------
    float
        Convection coefficient (W/(m²·K))
    """
    # Air properties at mean film temperature
    T_film = (T_surface + T_ambient) / 2
    
    # Approximate air properties (at ~400K)
    k_air = 0.035  # W/(m·K) - thermal conductivity
    nu_air = 2.5e-5  # m²/s - kinematic viscosity
    beta = 1 / T_film  # 1/K - volumetric thermal expansion coefficient for ideal gas
    g = 9.81  # m/s² - gravity
    
    # Rayleigh number
    Ra_L = (g * beta * abs(T_surface - T_ambient) * characteristic_length**3) / nu_air**2
    
    # Nusselt number correlation for vertical surfaces
    if orientation == 'vertical':
        # Churchill-Bernstein correlation for vertical surfaces
        Nu_L = 0.825 + (0.387 * Ra_L**(1/6)) / (1 + (0.492/0.0811)**(9/16))**(8/27)
    else:
        # Horizontal surface
        if Ra_L < 1e7:
            Nu_L = 1.36 * (Ra_L)**(1/5)
        else:
            Nu_L = 0.54 * (Ra_L)**(1/4)
    
    # Convection coefficient
    h_conv = (k_air / characteristic_length) * Nu_L
    
    return h_conv

# Test radiation at reference conditions
print("Testing radiation heat transfer:")
Q_rad_test = heat_radiation(600, 300, A_outer_outer, wall_emis())
print(f"  Box (600K) to surroundings (300K): {Q_rad_test:.2f} W")

Q_rad_enclosure = heat_radiation(1700, 300, A_graphite, gr_emis(1700), emis_cold=ins_emis())
print(f"  Graphite (1700K) to inner box (300K, enclosure): {Q_rad_enclosure:.2f} W")

h_conv_test = natural_convection_coefficient(500, 300, 2.0)
print(f"  Natural convection coefficient (500K, L=2m): {h_conv_test:.2f} W/(m²·K)")
print()

Testing radiation heat transfer:
  Box (600K) to surroundings (300K): 460809.86 W
  Graphite (1700K) to inner box (300K, enclosure): 5709044.03 W
  Natural convection coefficient (500K, L=2m): 0.30 W/(m²·K)



In [118]:

# ============================================================================
# SECTION 6: Graphite Shell Discretization (Spherical)
# ============================================================================

# Discretize graphite into spherical shells
N_SHELLS_GRAPHITE = 10  # Number of shells

# Calculate shell geometry
R_graphite = (3 * V_graphite / (4 * np.pi)) ** (1/3)  # Radius of graphite sphere (m)

# Create shell radii (from center to surface)
shell_radii = np.linspace(0, R_graphite, N_SHELLS_GRAPHITE + 1)

# Calculate shell volumes and interfacial areas
shell_volumes = np.zeros(N_SHELLS_GRAPHITE)
shell_areas = np.zeros(N_SHELLS_GRAPHITE + 1)  # Areas at interfaces (0 to N_SHELLS)

for i in range(N_SHELLS_GRAPHITE):
    r_inner = shell_radii[i]
    r_outer = shell_radii[i + 1]
    # Volume of spherical shell
    shell_volumes[i] = (4/3) * np.pi * (r_outer**3 - r_inner**3)
    # Area at inner radius of shell (interface between shell i-1 and i)
    if i == 0:
        shell_areas[0] = 0  # No area at center
    # Area at outer radius of shell (interface between shell i and i+1)
    shell_areas[i + 1] = 4 * np.pi * r_outer**2

# Calculate shell masses and thermal capacities
shell_masses = shell_volumes * rho_graphite_storage  # kg
shell_cp_array = np.array([gr_cp(T) for T in np.linspace(300, 1700, N_SHELLS_GRAPHITE)])  # J/(kg·K)
shell_thermal_capacities = shell_masses * shell_cp_array  # J/K

print(f"Graphite Shell Discretization:")
print(f"  Radius: {R_graphite:.4f} m")
print(f"  Number of shells: {N_SHELLS_GRAPHITE}")
print(f"  Total mass: {np.sum(shell_masses):.0f} kg (check: {m_graphite:.0f} kg)")
print(f"  Shell volumes (m³): {shell_volumes}")
print(f"  Shell surface areas (m²):")
for i in range(len(shell_areas)):
    print(f"    Interface {i}: {shell_areas[i]:.2f} m²")
print()



# ============================================================================
# SECTION 7: Heat Transfer Calculation with Proper Geometry
# ============================================================================

def update_graphite_shells(T_shells, Q_surface_in, dt, n_substeps=10):
    """
    Update graphite shell temperatures due to transient heat conduction.
    Uses internal substeps for numerical stability.
    
    T_shells: array of shell temperatures [0, ..., N_SHELLS_GRAPHITE-1]
    Q_surface_in: heat flow into the outermost shell from gap 1 (W)
    dt: time step (s)
    n_substeps: number of substeps to use internally (default 10)
    
    Returns:
    --------
    T_shells_new: updated shell temperatures after time step
    Q_inner_flow: array of heat flows between shells (for diagnostics)
    """
    
    dt_substep = dt / n_substeps
    T_shells_current = T_shells.copy()
    
    # Do n_substeps with same surface heat flow (more efficient and stable)
    for substep in range(n_substeps):
        T_shells_new = T_shells_current.copy()
        Q_flow = np.zeros(N_SHELLS_GRAPHITE)
        
        # Heat flow from gap 1 into outermost shell
        #Q_flow[N_SHELLS_GRAPHITE] = Q_surface_in
        
        # For each shell, calculate conduction to inner/outer neighbors
        for i in range(N_SHELLS_GRAPHITE):
            
            # Radial conduction to outer shell (or to surface at i = N_SHELLS_GRAPHITE - 1)
            if i == N_SHELLS_GRAPHITE - 1:
                # Outermost shell: heat flow out is Q_surface_in
                Q_out = -Q_surface_in
            else:
                # Heat conduction to next shell using spherical geometry
                #r_mid = (shell_radii[i+1] + shell_radii[i+2]) / 2  # Average radius for k
                T_mid = (T_shells_current[i] + T_shells_current[i+1]) / 2
                k_gr = gr_tcx(T_mid)  # W / (m K)
                
                # Spherical conduction: Q = 4π k (T_inner - T_outer) / (1/r_inner - 1/r_outer)
                r_i = shell_radii[i]  # Inner radius of boundary
                r_o = shell_radii[i+1]  # Outer radius of boundary
                
                Q_out = float(4 * np.pi * k_gr * (T_shells_current[i] - T_shells_current[i+1]) / 
                                (1.0/r_i - 1.0/r_o)) if (1.0/r_i - 1.0/r_o) != 0 else 0.0
            
            # Heat flow into this shell (from inner neighbor)
            if i == 0:
                # Center shell: no heat flow from inner (no inner boundary)
                Q_in = 0.0
            else:
                #r_mid = (shell_radii[i] + shell_radii[i+1]) / 2
                T_mid = (T_shells_current[i-1] + T_shells_current[i]) / 2
                k_gr = gr_tcx(T_mid)
                
                r_i = shell_radii[i]
                r_o = shell_radii[i+1]
                Q_in = float(4 * np.pi * k_gr * (T_shells_current[i-1] - T_shells_current[i]) / 
                            (1.0/r_i - 1.0/r_o)) if (1.0/r_i - 1.0/r_o) != 0 else 0.0
            
            Q_flow[i] = Q_in - Q_out

        #print(T_shells_current, Q_flow)    
        for i in range(N_SHELLS_GRAPHITE):
            # Update temperature
            if shell_thermal_capacities[i] > 0:
                dT = Q_flow[i] * dt_substep / shell_thermal_capacities[i]
                T_shells_new[i] = T_shells_current[i] + dT
        
        T_shells_current = T_shells_new
    
    return T_shells_current, Q_flow


def calculate_heat_flows_proper(T_gr_shells, T_inner_hot, T_inner_cold, T_outer_hot, T_outer_cold, T_ambient):
    """
    Quasi-steady state heat flow calculation using actual radiation and conduction.
    
    Given only the graphite surface temperature (outermost shell), iteratively solve 
    for all surface temperatures assuming continuity of heat flow at each interface.
    
    At each iteration:
    - Calculate actual radiation heat flow in gaps using heat_radiation()
    - Calculate conduction heat flow in insulation layers
    - Adjust surface temperatures to balance heat flow continuity
    """
    
    T_gr_surface = T_gr_shells[-1]
    
    # Iteratively solve for consistent surface temperatures
    # Initial guesses for insulation surface temperatures
    Q_iter = 1e5
    
    for iterate in range(50):
        Q_iter_last = Q_iter
        
        # ===== CALCULATE ACTUAL HEAT FLOWS AT CURRENT TEMPERATURES =====
        
        # Gap 1: Radiation between graphite surface and inner insulation hot side
        # Uses both surface emissivities in enclosure formula
        Q_gap1 = float(heat_radiation(T_gr_surface, T_inner_hot, A_graphite, 
                                       gr_emis(T_gr_surface), emis_cold=ins_emis()))
        R_gap1 = (T_gr_surface - T_inner_hot) / Q_gap1
        
        # Inner insulation: Conduction from hot to cold surface
        T_mean_inner = (T_inner_hot + T_inner_cold) / 2
        k_inner = ins_tcx(T_mean_inner)
        #Q_inner = float(k_inner * A_inner_outer * (T_inner_hot - T_inner_cold) / th_inner_insulation)
        R_inner = th_inner_insulation / k_inner / A_inner_outer
        
        # Gap 2: Radiation between inner insulation cold side and outer insulation hot side
        # Uses both surface emissivities in enclosure formula
        Q_gap2 = float(heat_radiation(T_inner_cold, T_outer_hot, A_outer_inner,
                                       ins_emis(), emis_cold=wall_emis()))
        R_gap2 = (T_inner_cold - T_outer_hot) / Q_gap2
        
        # Outer insulation: Conduction from hot to cold surface
        k_outer = wall_tcx()
        #Q_outer = float(k_outer * A_outer_outer * (T_outer_hot - T_outer_cold) / th_outer_insulation)
        R_outer = th_outer_insulation / k_outer / A_outer_outer
        # Ambient boundary: Radiation + convection
        h_conv = natural_convection_coefficient(T_outer_cold, T_ambient, L_outer_box, 'vertical')
        Q_conv = float(h_conv * A_outer_outer * (T_outer_cold - T_ambient))
        Q_rad_ambient = float(heat_radiation(T_outer_cold, T_ambient, A_outer_outer, wall_emis()))
        Q_ambient = Q_conv + Q_rad_ambient
        R_ambient = (T_outer_cold - T_ambient) / Q_ambient
        #print("resistances:",R_gap1, R_inner, R_gap2, R_outer, R_ambient)
        R_tot = R_gap1 + R_inner + R_gap2 + R_outer + R_ambient
        Q_iter = (T_gr_surface - T_ambient) / R_tot
        T_inner_hot = T_gr_surface - Q_iter * R_gap1
        T_inner_cold = T_inner_hot - Q_iter * R_inner
        T_outer_hot = T_inner_cold - Q_iter * R_gap2
        T_outer_cold = T_outer_hot - Q_iter * R_outer
        
        if abs(Q_iter - Q_iter_last) < 1e-3:
            break
    
    return {
        'T_gr_surface': float(T_gr_surface),
        'T_inner_hot': float(T_inner_hot),
        'T_inner_cold': float(T_inner_cold),
        'T_outer_hot': float(T_outer_hot),
        'T_outer_cold': float(T_outer_cold),
        'R_tot': float(R_tot),
        'Q_tot': float(Q_iter),
    }


# Test calculation at initial conditions
print("Testing heat flow calculation at initial conditions:")
T_shells_test = np.ones(N_SHELLS_GRAPHITE) * T_initial_graphite
flows_test = calculate_heat_flows_proper(T_shells_test, T_initial_inner_inner, T_initial_inner_outer, T_initial_outer_inner, T_initial_outer_outer, T_ambient)
print(f"  Graphite surface: {flows_test['T_gr_surface']-273.15:.1f}°C")
print(f"  Inner insulation: {flows_test['T_inner_hot']-273.15:.1f}°C → {flows_test['T_inner_cold']-273.15:.1f}°C")
print(f"  Outer insulation: {flows_test['T_outer_hot']-273.15:.1f}°C → {flows_test['T_outer_cold']-273.15:.1f}°C")
print(f"  Total heat flow: {flows_test['Q_tot']:.2f} W")

print()


Graphite Shell Discretization:
  Radius: 1.0883 m
  Number of shells: 10
  Total mass: 8100 kg (check: 8100 kg)
  Shell volumes (m³): [0.0054 0.0378 0.1026 0.1998 0.3294 0.4914 0.6858 0.9126 1.1718 1.4634]
  Shell surface areas (m²):
    Interface 0: 0.00 m²
    Interface 1: 0.15 m²
    Interface 2: 0.60 m²
    Interface 3: 1.34 m²
    Interface 4: 2.38 m²
    Interface 5: 3.72 m²
    Interface 6: 5.36 m²
    Interface 7: 7.29 m²
    Interface 8: 9.53 m²
    Interface 9: 12.06 m²
    Interface 10: 14.88 m²

Testing heat flow calculation at initial conditions:
  Graphite surface: 2399.8°C
  Inner insulation: 2398.8°C → 2250.4°C
  Outer insulation: 2249.8°C → 113.7°C
  Total heat flow: 57154.60 W



In [119]:
# Test calculation at initial conditions with shell model
print("Testing heat flow calculation at initial conditions (shell discretization):")

# Initialize graphite shells: all at initial temperature
T_shells = np.ones(N_SHELLS_GRAPHITE) * T_initial_graphite
flows_test = calculate_heat_flows_proper(T_shells, T_initial_inner_inner, T_initial_inner_outer, T_initial_outer_inner, T_initial_outer_outer, T_ambient)
print(f"  Graphite surface (outermost shell): {flows_test['T_gr_surface']-273.15:.1f}°C")
print(f"  Inner insulation: {flows_test['T_inner_hot']-273.15:.1f}°C → {flows_test['T_inner_cold']-273.15:.1f}°C")
print(f"  Outer insulation: {flows_test['T_outer_hot']-273.15:.1f}°C → {flows_test['T_outer_cold']-273.15:.1f}°C")

print()

# ============================================================================
# SECTION 8: Transient Simulation with Adaptive Time Stepping
# ============================================================================

# Final simulation time: 1 days
t_final = 1 * 24 * 3600  # seconds

# Storage for results
times = [0]
T_gr_center_history = [T_shells[0]]  # Average shell temperature
T_gr_surface_history = [T_shells[-1]]      # Outermost shell temperature
T_inner_hot_history = [T_initial_inner_inner]
T_inner_cold_history = [T_initial_inner_outer]
T_outer_hot_history = [T_initial_outer_inner]
T_outer_cold_history = [T_initial_outer_outer]

# Current state - only graphite shells are transient
T_shells = np.ones(N_SHELLS_GRAPHITE) * T_initial_graphite

t_current = 0
step = 0
dt = 600  # Start with 60 second time steps

print(f"Starting transient simulation to {t_final/(24*3600):.0f} days (quasi-steady insulation)")
print(f"Initial dt = {dt} seconds ({dt/60:.1f} minutes)")
print(f"Graphite shells: {N_SHELLS_GRAPHITE} discretization levels")
print(f"Insulation: Quasi-steady state (computed from thermal resistance network)\n")

while t_current < t_final:
    # Adaptive time stepping: increase dt as we go
    days_elapsed = t_current / (24 * 3600)
    
    if days_elapsed < 5:
        dt = 60 + (3600 - 60) * (days_elapsed / 5)
    elif days_elapsed < 30:
        dt = 3600 + (86400 - 3600) * ((days_elapsed - 5) / 25)
    else:
        dt = 86400
    
    if t_current + dt > t_final:
        dt = t_final - t_current
    
    # ===== SINGLE STEP: Calculate flows from thermal resistance network =====
    # Given only T_shells (graphite), compute surface temps and Q from resistance network
    flows = calculate_heat_flows_proper(T_shells, T_inner_hot_history[-1], T_inner_cold_history[-1], T_outer_hot_history[-1], T_outer_cold_history[-1], T_ambient)
    
    # ===== UPDATE GRAPHITE SHELLS WITH SUBSTEPS =====
    # Use the calculated Q_ambient as the heat loss driving the graphite cooling
    T_shells, Q_shell_flow = update_graphite_shells(T_shells, -flows['Q_tot'], dt, n_substeps=10)
    
    # ===== CLAMP TO AMBIENT =====
    T_shells = np.maximum(T_shells, T_ambient)
    
    # ===== UPDATE TIME AND STEP COUNTER =====
    t_current += dt
    step += 1
    
    # ===== STORE RESULTS IN HISTORY =====
    if step % 1 == 0 or abs(t_current - t_final) < 1:
        times.append(t_current)
        T_gr_center_history.append(np.mean(T_shells))
        T_gr_surface_history.append(T_shells[-1])
        T_inner_hot_history.append(flows['T_inner_hot'])
        T_inner_cold_history.append(flows['T_inner_cold'])
        T_outer_hot_history.append(flows['T_outer_hot'])
        T_outer_cold_history.append(flows['T_outer_cold'])
    
    # ===== PROGRESS OUTPUT =====
    T_avg_shells = np.mean(T_shells)
    
    print(f"Step {step:6d}: t={t_current/(24*3600):7.2f} days, dt={dt/3600:6.2f}h")
    print(f"  Graphite: avg={T_avg_shells-273.15:6.0f}°C, surface={T_shells[-1]-273.15:6.0f}°C")
    print(f"  Inner insulation: hot={flows['T_inner_hot']-273.15:5.0f}°C, cold={flows['T_inner_cold']-273.15:5.0f}°C")
    print(f"  Outer insulation: hot={flows['T_outer_hot']-273.15:5.0f}°C, cold={flows['T_outer_cold']-273.15:5.0f}°C")
    #print(f"  Heat flows [W]: Q_gap1={flows['Q_gap1']:8.1f}, Q_inner={flows['Q_inner']:8.1f}, Q_gap2={flows['Q_gap2']:8.1f}, Q_outer={flows['Q_outer']:8.1f}, Q_ambient={flows['Q_ambient']:8.1f}")
    print(f"  Heat flow: Q_tot={flows['Q_tot']:.1f} W")
    print(f"  Thermal resistance: R_tot={flows['R_tot']:.6f} K/W")

# Convert to arrays
times = np.array(times)
T_gr_center_history = np.array(T_gr_center_history)
T_gr_surface_history = np.array(T_gr_surface_history)
T_inner_hot_history = np.array(T_inner_hot_history)
T_inner_cold_history = np.array(T_inner_cold_history)
T_outer_hot_history = np.array(T_outer_hot_history)
T_outer_cold_history = np.array(T_outer_cold_history)
t_days = times / (24 * 3600)

print(f"\nSimulation complete!")
print(f"Final temperatures after {times[-1]/(24*3600):.1f} days:")
print(f"  Graphite average: {T_gr_center_history[-1]:.1f} K ({T_gr_center_history[-1]-273.15:.1f}°C)")
print(f"  Graphite surface: {T_gr_surface_history[-1]:.1f} K ({T_gr_surface_history[-1]-273.15:.1f}°C)")
print(f"  Inner insulation (hot): {T_inner_hot_history[-1]:.1f} K ({T_inner_hot_history[-1]-273.15:.1f}°C)")
print(f"  Inner insulation (cold): {T_inner_cold_history[-1]:.1f} K ({T_inner_cold_history[-1]-273.15:.1f}°C)")
print(f"  Outer insulation (hot): {T_outer_hot_history[-1]:.1f} K ({T_outer_hot_history[-1]-273.15:.1f}°C)")
print(f"  Outer insulation (cold): {T_outer_cold_history[-1]:.1f} K ({T_outer_cold_history[-1]-273.15:.1f}°C)")
print()

Testing heat flow calculation at initial conditions (shell discretization):
  Graphite surface (outermost shell): 2399.8°C
  Inner insulation: 2398.8°C → 2250.4°C
  Outer insulation: 2249.8°C → 113.7°C

Starting transient simulation to 1 days (quasi-steady insulation)
Initial dt = 600 seconds (10.0 minutes)
Graphite shells: 10 discretization levels
Insulation: Quasi-steady state (computed from thermal resistance network)

Step      1: t=   0.00 days, dt=  0.02h
  Graphite: avg=  2400°C, surface=  2399°C
  Inner insulation: hot= 2399°C, cold= 2250°C
  Outer insulation: hot= 2250°C, cold=  114°C
  Heat flow: Q_tot=57154.6 W
  Thermal resistance: R_tot=0.041519 K/W
Step      2: t=   0.00 days, dt=  0.02h
  Graphite: avg=  2400°C, surface=  2398°C
  Inner insulation: hot= 2398°C, cold= 2250°C
  Outer insulation: hot= 2249°C, cold=  114°C
  Heat flow: Q_tot=57134.7 W
  Thermal resistance: R_tot=0.041520 K/W
Step      3: t=   0.00 days, dt=  0.02h
  Graphite: avg=  2400°C, surface=  2398°C
 

In [ ]:
# ============================================================================
# SECTION 9: Results Visualization
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: All temperatures vs time (with hot/cold insulation surface details)
ax = axes[0, 0]
ax.plot(t_days, T_gr_center_history - 273.15, 'r-', linewidth=2.5, label='Graphite center (shell average)')
ax.plot(t_days, T_gr_surface_history - 273.15, 'r--', linewidth=2, label='Graphite surface (outermost shell)', alpha=0.8)
ax.plot(t_days, T_inner_hot_history - 273.15, 'b-', linewidth=2, label='Inner insulation (hot side)')
ax.plot(t_days, T_inner_cold_history - 273.15, 'b--', linewidth=2, label='Inner insulation (cold side)', alpha=0.7)
ax.plot(t_days, T_outer_hot_history - 273.15, 'orange', linewidth=2, label='Outer insulation (hot side)')
ax.plot(t_days, T_outer_cold_history - 273.15, color='orange', linewidth=2, linestyle='--', label='Outer insulation (cold side)', alpha=0.7)
ax.axhline(T_ambient - 273.15, color='k', linestyle='--', alpha=0.5, label='Ambient')
ax.set_xlabel('Time (days)', fontsize=11)
ax.set_ylabel('Temperature (°C)', fontsize=11)
ax.set_title('Transient Cooling: Shell Discretization + Hot/Cold Surfaces', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)

# Plot 2: Graphite cooling on log scale (shell model)
ax = axes[0, 1]
T_inner_mean_history = (T_inner_hot_history + T_inner_cold_history) / 2
T_outer_mean_history = (T_outer_hot_history + T_outer_cold_history) / 2

ax.semilogy(t_days, np.maximum(T_gr_center_history - T_ambient, 1), 'r-', linewidth=2.5, label='Graphite avg (shells)')
ax.semilogy(t_days, np.maximum(T_gr_surface_history - T_ambient, 1), 'r--', linewidth=2, label='Graphite surface', alpha=0.8)
ax.semilogy(t_days, np.maximum(T_inner_mean_history - T_ambient, 1), 'b-', linewidth=2, label='Inner insulation (mean)')
ax.semilogy(t_days, np.maximum(T_outer_mean_history - T_ambient, 1), 'orange', linewidth=2, label='Outer insulation (mean)')
ax.set_xlabel('Time (days)', fontsize=11)
ax.set_ylabel('Temperature above ambient (K, log)', fontsize=11)
ax.set_title('Cooling Rate (Log Scale) - Shell Model', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

# Plot 3: Temperature differences and gradients
ax = axes[1, 0]
dT_gr_to_inner_hot = T_gr_surface_history - T_inner_hot_history
dT_inner_to_outer = T_inner_cold_history - T_outer_hot_history
dT_outer_to_amb = T_outer_cold_history - T_ambient

ax.plot(t_days, dT_gr_to_inner_hot, 'r-', linewidth=2, label='ΔT graphite→inner hot (gap 1)')
ax.plot(t_days, dT_inner_to_outer, 'b-', linewidth=2, label='ΔT inner cold→outer hot (gap 2)')
ax.plot(t_days, dT_outer_to_amb, 'orange', linewidth=2, label='ΔT outer cold→ambient')
ax.plot(t_days, T_gr_surface_history - T_gr_center_history, 'purple', linewidth=2, linestyle='--', label='ΔT surface→average (graphite gradient)')
ax.plot(t_days, T_inner_hot_history - T_inner_cold_history, 'green', linewidth=1.5, linestyle=':', label='ΔT inner hot→cold')
ax.set_xlabel('Time (days)', fontsize=11)
ax.set_ylabel('Temperature difference (K)', fontsize=11)
ax.set_title('Thermal Resistance: Temperature Drops', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 4: Total energy removed (cumulative)
ax = axes[1, 1]
total_energy = np.zeros_like(times)
for i in range(1, len(times)):
    # Assuming constant Q over each timestep for approximation
    total_energy[i] = total_energy[i-1] + flows['Q_ambient'] * (times[i] - times[i-1]) / 1e9  # GJ

ax.plot(t_days, total_energy, 'g-', linewidth=2.5)
ax.fill_between(t_days, 0, total_energy, alpha=0.2, color='g')
ax.set_xlabel('Time (days)', fontsize=11)
ax.set_ylabel('Cumulative Energy to Ambient (GJ)', fontsize=11)
ax.set_title('Total Thermal Energy Dissipated', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTemperature gradients and thermal behavior:")
print(f"  Graphite surface: {T_gr_surface_history[-1]-273.15:.1f}°C")
print(f"  Graphite average: {T_gr_center_history[-1]-273.15:.1f}°C")
print(f"  Graphite gradient (center-surface): {(T_gr_center_history[-1]-T_gr_surface_history[-1]):.1f} K")
print(f"  Inner insulation gradient (hot-cold): {(T_inner_hot_history[-1]-T_inner_cold_history[-1]):.1f} K")
print(f"  Outer insulation gradient (hot-cold): {(T_outer_hot_history[-1]-T_outer_cold_history[-1]):.1f} K")
print()

In [ ]:

# ============================================================================
# SECTION 10: Analysis and Heat Flow Breakdown
# ============================================================================

# Find times to reach key temperatures
def find_time_to_temp(times, temps, T_target):
    """Find time when temperature reaches target."""
    idx = np.where(temps <= T_target)[0]
    if len(idx) > 0:
        return times[idx[0]]
    return None

print("=" * 80)
print("TIME TO REACH KEY TEMPERATURES")
print("=" * 80)

T_targets = [1500, 1400, 1300, 1200, 1000, 800, 600, 400]

print(f"\n{'Temperature':<15} {'Graphite':<20}")
print("-" * 35)

for T_tgt in T_targets:
    t_gr = find_time_to_temp(times, T_gr_center_history, T_tgt)
    t_gr_str = f"{t_gr/(24*3600):.2f} days" if t_gr else "N/A"
    print(f"{T_tgt-273.15:7.0f}°C        {t_gr_str:<20}")

# Energy analysis
print("\n" + "=" * 80)
print("THERMAL ENERGY ANALYSIS")
print("=" * 80)

E_initial = m_graphite * gr_cp(T_initial_graphite) * (T_initial_graphite - T_ambient)
E_final = m_graphite * gr_cp(T_gr_center_history[-1]) * (T_gr_center_history[-1] - T_ambient)
E_dissipated = E_initial - E_final

print(f"\nInitial thermal energy (graphite): {E_initial/3.6e9:.2f} MWh")
print(f"Final thermal energy (graphite): {E_final/3.6e9:.2f} MWh")
print(f"Total energy dissipated: {E_dissipated/3.6e9:.2f} MWh")
print(f"Average cooling power: {E_dissipated/times[-1]:.1f} W")

# Heat flow breakdown at key times
print("\n" + "=" * 80)
print("HEAT FLOW BREAKDOWN AT KEY TIMES")
print("=" * 80)

idx_points = [0, len(times)//4, len(times)//2, 3*len(times)//4, -1]

for idx in idx_points:
    # Create shell array for this timestep (use average as proxy)
    T_shells_idx = np.ones(N_SHELLS_GRAPHITE) * T_gr_center_history[idx]
    flows = calculate_heat_flows_proper(T_shells_idx, T_inner_hot_history[idx], T_inner_cold_history[idx], T_outer_hot_history[idx], T_outer_cold_history[idx])
    days = t_days[idx]
    
    print(f"\nt = {days:.1f} days (T_gr = {T_gr_center_history[idx]-273.15:.0f}°C)")
    print(f"  Graphite surface: {flows['T_gr_surface']-273.15:.1f}°C")
    print(f"  Inner insulation: {flows['T_inner_hot']-273.15:.1f}°C (hot) → {flows['T_inner_cold']-273.15:.1f}°C (cold)")
    print(f"  Outer insulation: {flows['T_outer_hot']-273.15:.1f}°C (hot) → {flows['T_outer_cold']-273.15:.1f}°C (cold)")
    print(f"\n  Heat flows:")
    print(f"    Graphite → Gap 1: {flows['Q_gap1']:.1f} W")
    print(f"    Through inner insulation: {flows['Q_inner']:.1f} W")
    print(f"    Through gap 2: {flows['Q_gap2']:.1f} W")
    print(f"    Through outer insulation: {flows['Q_outer']:.1f} W")

    print(f"    To ambient: {flows['Q_ambient']:.1f} W (conv: {flows['Q_conv']:.1f}W, rad: {flows['Q_rad']:.1f}W)")